In [3]:
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms

contrast_transforms = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ]
)


class ClfDS(Dataset):
    def __init__(self, folder):
        self.folder = folder
        classes=[]
        images=[]
        for cls in os.listdir(self.folder):
          for files in os.listdir(os.path.join(self.folder, cls)):
            img_path =os.path.join(self.folder, cls, files)
            images.append(img_path)
            classes.append(cls)
        self.id2label = {idx: label for idx, label in enumerate(sorted(set(classes)))}
        self.label2id = {label: idx for idx, label in enumerate(sorted(set(classes)))}
        self.classes = classes
        self.images = images

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        id=self.classes[idx]
        img = Image.open(img_path)
        img = contrast_transforms(img)

        return img, self.label2id[id]

In [4]:
from torch.utils.data import DataLoader
import os
train_dl = DataLoader(ClfDS(r'C:\Users\Administrator\Desktop\Coding\infaprim-clf-7\train'), batch_size=16, shuffle=True)
val_dl = DataLoader(ClfDS(r'C:\Users\Administrator\Desktop\Coding\infaprim-clf-7\valid'), batch_size=16, shuffle=False)

In [5]:
import math, json, os, torch, torch.nn as nn, torch.nn.functional as F
import pytorch_lightning as pl, timm
from pytorch_lightning.loggers import CSVLogger
from pytorch_lightning.callbacks import ModelCheckpoint
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

torch.set_float32_matmul_precision("high")
EPOCHS = 10          # keyin 100
N_CLS = len(set(train_dl.dataset.classes))


class ArcFaceLoss(nn.Module):
    def __init__(self, emb_dim, n_cls, s=30.0, m=0.5):
        super().__init__()
        self.n_cls, self.s, self.m = n_cls, s, m
        self.W = nn.Parameter(torch.empty(n_cls, emb_dim))
        nn.init.xavier_uniform_(self.W)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def cosine(self, emb):
        return F.linear(F.normalize(emb.float()), F.normalize(self.W.float()))

    def forward(self, emb, y):
        cos = self.cosine(emb).clamp(-1 + 1e-7, 1 - 1e-7)
        sin = torch.sqrt(1 - cos ** 2)
        phi = cos * math.cos(self.m) - sin * math.sin(self.m)
        phi = torch.where(cos > self.th, phi, cos - self.mm)
        onehot = F.one_hot(y, self.n_cls).bool()
        return F.cross_entropy(self.s * torch.where(onehot, phi, cos), y)


class LitArcFace(pl.LightningModule):
    def __init__(self, n_cls, lr=3e-4, emb_dim=128):
        super().__init__()
        self.save_hyperparameters()
        self.backbone = timm.create_model("vit_tiny_patch16_224", pretrained=True, num_classes=0)
        self.proj = nn.Sequential(nn.Linear(192, emb_dim), nn.BatchNorm1d(emb_dim))
        self.crit = ArcFaceLoss(emb_dim, n_cls)
        self._p, self._t = [], []

    def forward(self, x):
        return self.proj(self.backbone(x))

    def training_step(self, batch, _):
        x, y = batch
        loss = self.crit(self(x), y)
        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        return loss

    def validation_step(self, batch, _):
        x, y = batch
        emb = self(x)
        self.log("val_loss", self.crit(emb, y), prog_bar=True)
        self._p.append(self.crit.cosine(emb).argmax(1).cpu())   # eng katta cosine
        self._t.append(y.cpu())

    def on_validation_epoch_end(self):
        p, t = torch.cat(self._p).numpy(), torch.cat(self._t).numpy()
        self._p.clear(); self._t.clear()
        pm, rm, fm, _ = precision_recall_fscore_support(t, p, average="macro", zero_division=0)
        pw, rw, fw, _ = precision_recall_fscore_support(t, p, average="weighted", zero_division=0)
        self.log_dict({"val_acc": accuracy_score(t, p),
                       "val_prec_macro": pm, "val_rec_macro": rm, "val_f1_macro": fm,
                       "val_prec_w": pw, "val_rec_w": rw, "val_f1_w": fw})

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=0.05)
        return [opt], [torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.trainer.max_epochs)]


pl.seed_everything(42)
model = LitArcFace(N_CLS)
ckpt = ModelCheckpoint(monitor="val_f1_macro", mode="max")
trainer = pl.Trainer(max_epochs=EPOCHS, accelerator="gpu", precision="16-mixed",
                     logger=CSVLogger("logs", name=f"arcface_{EPOCHS}ep"),
                     callbacks=[ckpt], log_every_n_steps=1)
trainer.fit(model, train_dl, val_dl)

os.makedirs("results", exist_ok=True)
res = {k: float(v) for k, v in trainer.callback_metrics.items() if k.startswith("val_")}
res["best_f1_macro"] = float(ckpt.best_model_score)
json.dump(res, open(f"results/arcface_{EPOCHS}ep.json", "w"), indent=2)
print(res)

W1027 16:40:51.765000 29276 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Seed set to 42
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\pytorch_lightning\utilities\model_summary\model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name     ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ backbone │ VisionTransformer │  5.5 M │ train │     0 │
│ 1 │ proj     │ Sequential        │ 25.0 K │ train │     0 │
│ 2 │ crit     │ ArcFaceLoss       │ 22.0 K │ train │     0 │
└───┴──────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 5.6 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 5.6 M                                                                                                
Total estimated model params size (MB): 22.286                                                                     
Modules in train mode: 280                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.p
y:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of 
the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.

c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\pytorch_lightning\utilities\_pytree.py:21: 
`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.p
y:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value 
of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.

`Trainer.fit` stopped: `max_epochs=10` reached.


{'val_loss': 3.931424379348755, 'val_acc': 0.8361632823944092, 'val_prec_macro': 0.7719159126281738, 'val_rec_macro': 0.7568864822387695, 'val_f1_macro': 0.7560807466506958, 'val_prec_w': 0.8373489379882812, 'val_rec_w': 0.8361632823944092, 'val_f1_w': 0.8345205187797546, 'best_f1_macro': 0.7560807466506958}


In [7]:
torch.save(model, "ArcFace_loss_VIT_clf.pth")

In [9]:
model=torch.load(r"C:\Users\Administrator\Desktop\Coding\ArcFace_loss_VIT_clf.pth", weights_only=False)
model.eval()

LitArcFace(
  (backbone): VisionTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
      (norm): Identity()
    )
    (pos_drop): Dropout(p=0.0, inplace=False)
    (patch_drop): Identity()
    (norm_pre): Identity()
    (blocks): Sequential(
      (0): Block(
        (norm1): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=192, out_features=576, bias=True)
          (q_norm): Identity()
          (k_norm): Identity()
          (attn_drop): Dropout(p=0.0, inplace=False)
          (norm): Identity()
          (proj): Linear(in_features=192, out_features=192, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (ls1): Identity()
        (drop_path1): Identity()
        (norm2): LayerNorm((192,), eps=1e-06, elementwise_affine=True)
        (mlp): Mlp(
          (fc1): Linear(in_features=192, out_features=768, bias=True)
          (ac

In [10]:
clf_model = model.cuda()
import torch
import torch.nn.functional as F

from sklearn.metrics import classification_report
import numpy as np

all_preds = []
all_labels = []

for batch in val_dl:
    images, labels = batch
    val_out = clf_model(images.cuda())

    probs = F.softmax(val_out, dim=-1)
    preds = torch.argmax(probs, dim=-1).cpu().numpy()
    labels = labels.cpu().numpy()

    preds = [val_dl.dataset.id2label[idx] for idx in preds]
    labels = [val_dl.dataset.id2label[idx] for idx in labels]

    all_preds.extend(preds)
    all_labels.extend(labels)

print(classification_report(all_labels, all_preds))

                          precision    recall  f1-score   support

                n_1_1050       0.00      0.00      0.00        17
              n_1_1050-2       0.00      0.00      0.00        27
              n_1_1050-3       0.00      0.00      0.00       139
                 n_1_300       0.00      0.00      0.00        79
               n_1_300-2       0.00      0.00      0.00        71
               n_1_300-3       0.03      0.01      0.01       182
                 n_1_600       0.00      0.00      0.00        40
               n_1_600-2       0.14      0.10      0.12        48
               n_1_600-3       0.00      0.00      0.00       163
                n_2_1050       0.00      0.00      0.00         9
              n_2_1050-2       0.00      0.00      0.00        32
              n_2_1050-3       0.00      0.00      0.00       106
                 n_2_300       0.00      0.00      0.00        81
               n_2_300-2       0.00      0.00      0.00        65
         

c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Administrator\Desktop\Coding\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital